# KAE Multi-Model Agent (Colab)

Один Colab-GPU агент обслуживает несколько задач и **переключает модели по API**
(ленивая загрузка под задачу, экономия VRAM). KAE роутит запросы по ролям агента:

| task | модель | роль в KAE |
|---|---|---|
| `table`   | GOT-OCR2.0            | table   |
| `formula` | GOT-OCR2.0            | formula |
| `vision`  | Qwen2.5-VL-7B (4bit)  | vision  |

HTTP: `GET /health` (для менеджера агентов) и `POST /infer {image_b64, task, prompt?}`.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU** (не TPU).

In [ ]:
# 1. GPU + зависимости
!nvidia-smi -L || print('Поставь T4 GPU: Runtime -> Change runtime type -> T4 GPU')
!pip -q install transformers==4.49.0 tiktoken==0.6.0 verovio accelerate \
    qwen-vl-utils bitsandbytes torchvision flask

In [ ]:
# 2. Реестр моделей с ленивой загрузкой (грузим модель при первом запросе задачи)
import torch, gc
_CACHE = {}

def _free_vram():
    gc.collect(); torch.cuda.empty_cache()

def get_got_ocr():
    if 'got' not in _CACHE:
        from transformers import AutoModel, AutoTokenizer
        tok = AutoTokenizer.from_pretrained('stepfun-ai/GOT-OCR2_0', trust_remote_code=True)
        m = AutoModel.from_pretrained('stepfun-ai/GOT-OCR2_0', trust_remote_code=True,
            low_cpu_mem_usage=True, device_map='cuda', use_safetensors=True,
            pad_token_id=tok.eos_token_id).eval().cuda()
        _CACHE['got'] = (m, tok)
    return _CACHE['got']

def get_qwen_vl():
    if 'qwen' not in _CACHE:
        from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        m = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            'Qwen/Qwen2.5-VL-7B-Instruct', quantization_config=bnb, device_map='cuda')
        proc = AutoProcessor.from_pretrained('Qwen/Qwen2.5-VL-7B-Instruct')
        _CACHE['qwen'] = (m, proc)
    return _CACHE['qwen']

print('Реестр моделей готов (GOT-OCR: table/formula, Qwen2.5-VL: vision)')

In [ ]:
# 3. Инференс по задаче
def infer(image_path, task='table', prompt=None):
    if task in ('table', 'formula', 'ocr'):
        m, tok = get_got_ocr()
        return m.chat(tok, image_path, ocr_type='format')
    if task == 'vision':
        from qwen_vl_utils import process_vision_info
        m, proc = get_qwen_vl()
        prompt = prompt or ('Reconstruct this block diagram as a LaTeX tikzpicture: '
                            'boxes with their text, titles, and connecting arrows. '
                            'Output only the tikzpicture environment.')
        msgs = [{'role': 'user', 'content': [
            {'type': 'image', 'image': image_path}, {'type': 'text', 'text': prompt}]}]
        text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        imgs, vids = process_vision_info(msgs)
        inputs = proc(text=[text], images=imgs, videos=vids, padding=True, return_tensors='pt').to('cuda')
        out = m.generate(**inputs, max_new_tokens=2048)
        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
        return proc.batch_decode(trimmed, skip_special_tokens=True)[0]
    raise ValueError(f'unknown task: {task}')

print('infer() готов')

In [ ]:
# 4. HTTP-сервис: /health + /infer
from flask import Flask, request, jsonify
import base64, tempfile, threading, os
app = Flask(__name__)

@app.get('/health')
def health():
    return jsonify({'status': 'ok', 'kind': 'multimodel',
                    'tasks': ['table', 'formula', 'vision'],
                    'loaded': list(_CACHE.keys())})

@app.post('/infer')
def _infer():
    d = request.get_json(force=True)
    raw = base64.b64decode(d['image_b64'])
    fd, p = tempfile.mkstemp(suffix='.png'); os.write(fd, raw); os.close(fd)
    try:
        text = infer(p, task=d.get('task', 'table'), prompt=d.get('prompt'))
    finally:
        os.remove(p)
    return jsonify({'text': text})

# /ocr — совместимость со старым GOT-OCR агентом
@app.post('/ocr')
def _ocr():
    d = request.get_json(force=True)
    raw = base64.b64decode(d['image_b64'])
    fd, p = tempfile.mkstemp(suffix='.png'); os.write(fd, raw); os.close(fd)
    try:
        text = infer(p, task='table')
    finally:
        os.remove(p)
    return jsonify({'text': text})

PORT = 5005
threading.Thread(target=lambda: app.run(host='0.0.0.0', port=PORT), daemon=True).start()
print(f'Сервис на :{PORT} (/health, /infer, /ocr)')

In [ ]:
# 5. Туннель наружу — публичный URL для менеджера агентов KAE
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess, re, itertools
proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in itertools.islice(proc.stdout, 200):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\n' + '=' * 60)
print('AGENT URL:', url)
print('Добавь в менеджер агентов KAE (kind=multimodel). Роли: table, formula, vision')
print('=' * 60)
import time
while True: time.sleep(300)